# Outline
Node classification & edge prediction (Multitask learning)
Text encoder - all-MiniLM-L6-v2 from sentence-transformers

Architecture modification
Data Processing, GAT model, training loop, evaluation on test data

In [1]:
import sys
import torch
print(f"Python version: {sys.version}")
print(f"Path to interpreter: {sys.executable}")
print(f"Version of PyTorch: {torch.__version__}")
print(f"CUDA is available: {torch.cuda.is_available()}")
print(f"MPS is available: {torch.backends.mps.is_available()}")
if not torch.cuda.is_available() and not torch.backends.mps.is_available():
    import os
    print("\nDiagnosis:")
    if "cpu" in torch.__version__:
        print("-> GPU installation failed.")
    else:
        print("-> GPU installed but does not see card. Check nvidia driver.")

Python version: 3.12.13 (main, Mar 25 2026, 03:16:06) [Clang 22.1.1 ]
Path to interpreter: /Users/zrutix/Projects/Bakalarka/gat-document-relation/.venv/bin/python
Version of PyTorch: 2.11.0
CUDA is available: False
MPS is available: True


In [2]:
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import os
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

/Users/zrutix/Projects/Bakalarka/gat-document-relation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Build Node Features

In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np, torch, re, unicodedata

miniLM = SentenceTransformer('all-MiniLM-L6-v2')
for p in miniLM.parameters(): p.requires_grad = False

def remove_diacritics(text):
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

def extract_manual_text_features(text):
    t = text.strip(); tl = remove_diacritics(t.lower())
    is_fig = 1.0 if re.match(r'^(figure|fig\.?|obrazok|obr\.?)\s*[\d\(\[]', tl) else 0.0
    is_tab = 1.0 if re.match(r'^(table|tab\.?|tabulka)\s*[\d\(\[]', tl) else 0.0
    is_num = 1.0 if re.match(r'^\(?\d+[\.\)\:]', tl) else 0.0
    words = t.split(); wc = len(words)
    is_short = 1.0 if wc < 20 else 0.0
    len_norm = min(wc / 100.0, 1.0)
    return [is_fig, is_tab, is_num, is_short, len_norm]

def build_node_features(graph, iou_threshold=0.5):
    W, H = graph['width'], graph['height']
    ann_nodes, yolo_nodes = graph['nodes'], graph['yolo_nodes']

    # --- IoU matching (greedy 1-to-1) ---
    pairs = []
    for yi, yn in enumerate(yolo_nodes):
        for ai, ann in enumerate(ann_nodes):
            s = iou(ann['bbox'], yn['geometry']['absolute_pixel_coords'])
            if s >= iou_threshold:
                pairs.append((s, yi, ai))
    pairs.sort(reverse=True)

    matched_yi, matched_ai, assignments = set(), set(), {}
    for s, yi, ai in pairs:
        if yi in matched_yi or ai in matched_ai: continue
        assignments[yi] = ann_nodes[ai]
        matched_yi.add(yi); matched_ai.add(ai)

    feat_geom, feat_yolo, feat_text_manual, texts, matched_node_ids = [], [], [], [], []
    
    for yi, yn in enumerate(yolo_nodes):
        ann = assignments.get(yi)
        if ann is None: continue

        x1, y1, x2, y2 = yn['geometry']['absolute_pixel_coords']
        w, h = x2 - x1, y2 - y1

        # 11-dim geom — identical with pipeline
        x1n, y1n, x2n, y2n = x1/W, y1/H, x2/W, y2/H
        xc, yc = (x1n+x2n)/2, (y1n+y2n)/2
        wn, hn = x2n-x1n, y2n-y1n
        area = np.log(((w*h)/(W*H)) + 1e-6)
        aspect = np.log((w/(h+1e-6)) + 1e-6)
        conf = float(yn.get('yolo_confidence', 1.0))
        feat_geom.append([x1n,y1n,x2n,y2n, xc,yc, wn,hn, area, aspect, conf])

        # 11-dim YOLO soft-label — from yolo
        cls = yn.get('label_id', None)
        soft = np.full(11, (1.0-conf)/10.0, dtype=np.float32)
        if cls is not None and 0 <= int(cls) < 11:
            soft[int(cls)] = conf
        else:
            soft = np.full(11, 1.0/11.0, dtype=np.float32)
        soft *= 0.3
        feat_yolo.append(soft.tolist())

        # Text: 5 manual + 384 MiniLM
        txt = yn.get('text', '')
        texts.append(txt)
        feat_text_manual.append(extract_manual_text_features(txt))
        matched_node_ids.append(ann['node_id'])

    if len(matched_node_ids) == 0:
        return (
            torch.empty((0, 11),  dtype=torch.float),
            torch.empty((0, 11),  dtype=torch.float),
            torch.empty((0, 389), dtype=torch.float),
            []
        )

    with torch.no_grad():
        emb = miniLM.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

    feat_geom = torch.tensor(feat_geom, dtype=torch.float)
    feat_yolo = torch.tensor(feat_yolo, dtype=torch.float)
    feat_text = torch.tensor(np.concatenate([np.array(feat_text_manual), emb], axis=1), dtype=torch.float)

    return feat_geom, feat_yolo, feat_text, matched_node_ids

# knn Graph construction

In [13]:
def build_knn_edges(features, k=8):
    num_nodes = features.shape[0]
    if num_nodes <= 1:
        return torch.empty((2, 0), dtype=torch.long)
    actual_k = min(k, num_nodes - 1)
    
    if torch.is_tensor(features):
        centers = features[:, 4:6].numpy()
    else:
        centers = features[:, 4:6]
    
    edge_list = []
    for i in range(num_nodes):
        dx = centers[:, 0] - centers[i, 0]
        dy = centers[:, 1] - centers[i, 1]
        distances = np.sqrt(dx**2 + (1.5 * dy)**2)
        nearest = np.argsort(distances)[1:actual_k+1]
        for j in nearest:
            edge_list.append([i, j])
            edge_list.append([j, i])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_index = torch.unique(edge_index, dim=1)
    return edge_index

# Build edge targets

In [14]:
def build_edge_targets(edges, matched_node_ids, id2cat, class_map):
    id_map  = {ann_id: idx for idx, ann_id in enumerate(matched_node_ids)}
    targets = {}
 
    for edge in edges:
        if edge['from'] not in id_map or edge['to'] not in id_map:
            continue
        if edge['label'] != 1:
            continue
 
        si = id_map[edge['from']]
        di = id_map[edge['to']]
        s_cls = class_map.get(id2cat.get(edge['from']), 7)
        d_cls = class_map.get(id2cat.get(edge['to']),   7)
        pair  = frozenset([s_cls, d_cls])
 
        if pair == frozenset([0, 1]):   # caption–figure
            targets[(si, di)] = 1
            targets[(di, si)] = 1       # obojsmerné — GAT je undirected
        elif pair == frozenset([0, 2]): # caption–table
            targets[(si, di)] = 2
            targets[(di, si)] = 2
 
    return targets

# load data

In [15]:
CLASS_MAP = {
    1: 0,   # Caption
    7: 1,   # Picture
    9: 2,   # Table
    3: 3,   # Formula
    8: 4,   # Section-header
    5: 5,   # Page-footer
    6: 6,   # Page-header
    2: 7, 4: 7, 10: 7, 11: 7,   # Footnote, List-item, Text, Title -> Other
}

def load_all_data(folder):
    out = []
 
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith('.json'):
            continue
        with open(os.path.join(folder, fn), encoding='utf-8') as f:
            g = json.load(f)
 
        feat_geom, feat_yolo, feat_text, matched_ids = build_node_features(g)
        if feat_geom.shape[0] < 2:
            continue
 
        # KNN graph
        edge_index = build_knn_edges(feat_geom, k=8)
 
        id2cat  = {n['node_id']: n['category_id'] for n in g['nodes']}
        y_nodes = torch.tensor(
            [CLASS_MAP[id2cat[nid]] for nid in matched_ids],
            dtype=torch.long,
        )
 
        edge_targets_dict = build_edge_targets(g['edges'], matched_ids, id2cat, CLASS_MAP)
 
        # Candidate edges: Caption × (Picture | Table)
        cap_idx     = (y_nodes == 0).nonzero(as_tuple=True)[0]
        partner_idx = torch.cat([
            (y_nodes == 1).nonzero(as_tuple=True)[0],  # Picture
            (y_nodes == 2).nonzero(as_tuple=True)[0],  # Table
        ])
 
        if len(cap_idx) > 0 and len(partner_idx) > 0:
            # Cartesian: caption x each partner
            src = cap_idx.repeat_interleave(len(partner_idx))
            dst = partner_idx.repeat(len(cap_idx))
            edge_index_targets = torch.stack([src, dst], dim=0)  # [2, E]
 
            # For each candidate edge: 0=no-rel, 1=cap-fig, 2=cap-tab
            y_edges = torch.zeros(edge_index_targets.shape[1], dtype=torch.long)
            for i in range(edge_index_targets.shape[1]):
                s = edge_index_targets[0, i].item()
                d = edge_index_targets[1, i].item()
                y_edges[i] = edge_targets_dict.get((s, d), 0)
        else:
            edge_index_targets = torch.empty((2, 0), dtype=torch.long)
            y_edges            = torch.empty((0,),   dtype=torch.long)
 
        out.append(Data(
            feat_geom=feat_geom,
            feat_yolo=feat_yolo,
            feat_text=feat_text,
            edge_index=edge_index,
            y_nodes=y_nodes,
            edge_index_targets=edge_index_targets,
            y_edges=y_edges
        ))
 
    return out

In [29]:
def iou(a_xywh, b_xyxy):
    ax1, ay1 = a_xywh[0], a_xywh[1]
    ax2, ay2 = a_xywh[0] + a_xywh[2], a_xywh[1] + a_xywh[3]
    bx1, by1, bx2, by2 = b_xyxy

    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    area_a = a_xywh[2] * a_xywh[3]
    area_b = (bx2 - bx1) * (by2 - by1)
    union  = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

# Data loader

In [32]:
SAVE_DIR = "data/DocLayNet/processed_data"
os.makedirs(SAVE_DIR, exist_ok=True)

train_path = os.path.join(SAVE_DIR, "train_graphs.pt")
val_path = os.path.join(SAVE_DIR, "val_graphs.pt")
test_path = os.path.join(SAVE_DIR, "test_graphs.pt")

if os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path):
    print("Loading already preprocessed data from disk...")
    train_graphs = torch.load(train_path, weights_only=False)
    val_graphs = torch.load(val_path, weights_only=False)
    test_graphs = torch.load(test_path, weights_only=False)

else:
    print("Data not saved. Starting preprocessing...")
    train_graphs = load_all_data("data/DocLayNet/train_data")
    val_graphs = load_all_data("data/DocLayNet/val_data")
    test_graphs = load_all_data("data/DocLayNet/test_data")
    
    print("Saving processed data on the disk...")
    torch.save(train_graphs, train_path)
    torch.save(val_graphs, val_path)
    torch.save(test_graphs, test_path)

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_graphs, batch_size=32, shuffle=False)

print(f"Data are ready: Train({len(train_graphs)}), Val({len(val_graphs)}), Test({len(test_graphs)})")

Data not saved. Starting preprocessing...
Saving processed data on the disk...
Data are ready: Train(18812), Val(9135), Test(6766)


# GAT Model

In [75]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

_model = None

class DocumentMultiTaskGAT(nn.Module):
    def __init__(self, num_node_classes=8):
        super().__init__()
        self.geom_proj = nn.Linear(11, 64)
        self.yolo_proj = nn.Linear(11, 32)
        # droupout for yolo features to learn reaclassify
        self.yolo_dropout = nn.Dropout(0.2)
        self.text_proj = nn.Linear(389, 160)

        self.conv1 = GATv2Conv(256, 64, heads=4, concat=True, dropout=0.2)
        self.conv2 = GATv2Conv(256, 64, heads=4, concat=True, dropout=0.2)
        self.conv3 = GATv2Conv(256, 32, heads=8, concat=True, dropout=0.2)

        # Correction gate: to learn when to ignore YOLO
        self.correction_gate = nn.Sequential(
            nn.Linear(256 + 11, 64),  # GAT embedding + yolo soft label
            nn.ELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()  # 0 = trust YOLO, 1 = trust GAT
        )

        self.node_classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_node_classes)
        )
        self.edge_classifier = nn.Sequential(
            nn.Linear(512 + 6, 128),
            nn.ELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3)
        )

    def forward(self, batch):
        x_geom = batch.feat_geom
        x_yolo = batch.feat_yolo
        x_text = batch.feat_text
    
        h_geom = F.elu(self.geom_proj(x_geom))
        h_yolo = F.elu(self.yolo_proj(x_yolo))
        h_yolo = self.yolo_dropout(h_yolo)
        h_text = F.elu(self.text_proj(x_text))
    
        x = torch.cat([h_geom, h_yolo, h_text], dim=-1)
        x_res1 = x
        x = F.elu(self.conv1(x, batch.edge_index)) + x_res1
        x_res2 = x
        x = F.elu(self.conv2(x, batch.edge_index)) + x_res2
        z = F.elu(self.conv3(x, batch.edge_index))
    
        gate = self.correction_gate(
            torch.cat([z, x_yolo], dim=-1)
        )  # [N, 1]
    
        gat_logits = self.node_classifier(z)  # [N, 8]
    
        # YOLO:  0    1    2    3    4    5    6    7    8    9    10
        #       Cap  Fn   Frm  Li   PgF  PgH  Pic  SecH Tab  Txt  Tit
        YOLO_TO_8 = torch.tensor(
            [0,   7,   3,   7,   5,   6,   1,   4,   2,   7,   7],
            dtype=torch.long, device=x_yolo.device
        )
        yolo_8 = torch.zeros(x_yolo.shape[0], 8, device=x_yolo.device)
        for yolo_idx in range(11):
            gat_idx = YOLO_TO_8[yolo_idx]
            yolo_8[:, gat_idx] = yolo_8[:, gat_idx] + x_yolo[:, yolo_idx]
    
        yolo_pred_class = x_yolo.argmax(dim=-1)  # [N], hodnoty 0–10
    
        is_yolo_table = (yolo_pred_class == 8).float().unsqueeze(-1)  # [N, 1]
        gate_effective = gate * (1.0 - is_yolo_table)
    
        node_logits = gate_effective * gat_logits + (1.0 - gate_effective) * yolo_8
    
        node_logits = node_logits.clone()
        node_logits[yolo_pred_class != 8, 2] = -1e9
    
        return z, node_logits, gate

    def predict_edges(self, z, query_edge_index, feat_geom=None):
        src_z = z[query_edge_index[0]]
        dst_z = z[query_edge_index[1]]
        edge_feat = torch.cat([src_z, dst_z], dim=-1)

        if feat_geom is not None:
            src_g = feat_geom[query_edge_index[0]][:, :6]  # x1,y1,x2,y2,xc,yc
            dst_g = feat_geom[query_edge_index[1]][:, :6]
            edge_feat = torch.cat([edge_feat, src_g - dst_g], dim=-1)

        return self.edge_classifier(edge_feat)  # [E, 3] logity


# Node weight calculation

In [71]:
from collections import Counter
cnt = Counter()
for g in train_graphs:
    cnt.update(g.y_nodes.tolist())
print(cnt)

import numpy as np
counts = np.array([cnt[i] for i in range(8)])
sqrt_inv = np.sqrt(counts.sum() / counts)
sqrt_inv_norm = sqrt_inv / sqrt_inv[7]    # normalize so Other (index 7) = 1.0
print("Weights:", sqrt_inv_norm.tolist())

Counter({7: 115227, 4: 26593, 1: 19878, 6: 15617, 2: 14554, 5: 13946, 0: 11194, 3: 3692})
Weights: [3.2083702170580626, 2.4076357553784, 2.813752812904125, 5.586583332065361, 2.081581697599194, 2.8744337073768316, 2.716303681938853, 1.0]


In [78]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix
import random

RUN_NAME = 'run_yolo_augmentation6'
OUT_DIR  = os.path.join('models', 'gat_graph_text', RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)


if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    
device = torch.device('cpu')
print("USING:", device)

model = DocumentMultiTaskGAT(num_node_classes=8).to(device)


node_weights = torch.tensor(
    [3.2083702170580626, 2.4076357553784, 4.0, 5.586583332065361, 2.081581697599194, 2.8744337073768316, 2.716303681938853, 1.0],
    #   0     1    2    3    4    5    6    7
    # Caption Pic Table Form SecH PgF  PgH  Other
    dtype=torch.float
).to(device)

node_criterion = nn.CrossEntropyLoss(weight=node_weights)
edge_criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 4.0, 4.0]).to(device)
)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
 
def apply_targeted_yolo_noise(feat_yolo, y_nodes=None, p_default=0.10, p_table=0.35):
    """
    feat_yolo: [N, 11] soft labels from YOLO.
    Smart swap based on real yolo mistakes + more aggressive noise for Tables.
    """
    noisy_yolo = feat_yolo.clone()
    N = noisy_yolo.shape[0]

    swap_map = {   
        0: [7, 9],    # Caption → Section-header or Text
        7: [0, 10],   # Section-header → Caption or Title
        9: [0, 3],    # Text → Caption or List-item
    }

    for i in range(N):
        cls = y_nodes[i].item() if y_nodes is not None else -1
        p = p_table if cls == 2 else p_default

        if torch.rand(1).item() < p:
            current_class = noisy_yolo[i].argmax().item()
            if current_class in swap_map:
                target_class = random.choice(swap_map[current_class])
                temp = noisy_yolo[i, current_class].clone()
                noisy_yolo[i, current_class] = noisy_yolo[i, target_class]
                noisy_yolo[i, target_class] = temp

    return noisy_yolo


def train():
    model.train()
    total_loss = 0
    processed  = 0

    for batch in train_loader:
        batch = batch.to(device)

        # Augmentation — Table gets more aggresive noise (p_table=0.35)
        batch.feat_yolo = apply_targeted_yolo_noise(
            batch.feat_yolo,
            y_nodes=batch.y_nodes,
            p_default=0.10,
            p_table=0.35,
        )

        optimizer.zero_grad()

        z, node_logits, gate = model(batch)

        loss_node = node_criterion(node_logits, batch.y_nodes)

        # Gate regularization: pushes gate to unsure values (not 0 or 1)
        gate_entropy = -(
            gate * torch.log(gate + 1e-8) +
            (1.0 - gate) * torch.log(1.0 - gate + 1e-8)
        ).mean()
        loss_gate = -0.01 * gate_entropy

        if batch.edge_index_targets.shape[1] > 0:
            edge_logits = model.predict_edges(
                z, batch.edge_index_targets, batch.feat_geom
            )
            loss_edge = edge_criterion(edge_logits, batch.y_edges)
        else:
            loss_edge = torch.tensor(0.0, device=device)

        loss = loss_node + 0.5 * loss_edge + loss_gate
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        processed  += 1

    return total_loss / processed if processed > 0 else 0
 
 
# EVALUATE LOSS
 
def evaluate_loss(loader):
    model.eval()
    total_loss = 0
    processed  = 0
 
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            z, node_logits, _ = model(batch)
 
            loss_node = node_criterion(node_logits, batch.y_nodes)
 
            if batch.edge_index_targets.shape[1] > 0:
                edge_logits = model.predict_edges(
                    z, batch.edge_index_targets, batch.feat_geom
                )
                loss_edge = edge_criterion(edge_logits, batch.y_edges)
            else:
                loss_edge = torch.tensor(0.0, device=device)
 
            total_loss += (loss_node + 0.5 * loss_edge).item()
            processed  += 1
 
    return total_loss / processed if processed > 0 else 0
 
 
# TRAINING LOOP
 
STOPPAGE      = 7
best_val_loss = float('inf')
worse_count   = 0
epoch         = 1
train_loss_history = []
val_loss_history   = []
 
print(f'\nSaving checkpoints to: {OUT_DIR}\n')
 
while worse_count < STOPPAGE:
    train_loss = train()
    val_loss   = evaluate_loss(val_loader)
 
    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
 
    checkpoint_path = os.path.join(
        OUT_DIR,
        f'epoch_{epoch:03d}_tl_{train_loss:.4f}_vl_{val_loss:.4f}.pt',
    )
    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss':           train_loss,
        'val_loss':             val_loss,
    }, checkpoint_path)
 
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        worse_count   = 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR, 'best_model.pt'))
        print(f'Epoch {epoch:03d} | train: {train_loss:.4f} | val: {val_loss:.4f} | ✓ NEW BEST')
    else:
        worse_count += 1
        print(f'Epoch {epoch:03d} | train: {train_loss:.4f} | val: {val_loss:.4f} | worse ({worse_count}/{STOPPAGE})')
 
    epoch += 1
 
print(f'\nTraining done. Best val loss: {best_val_loss:.4f}')
 
 
# PLOT LOSSES
 
def plot_losses(train_history, val_history, out_dir):
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_history) + 1)
    plt.plot(epochs, train_history, 'b-', label='Training Loss',   linewidth=2)
    plt.plot(epochs, val_history,   'r-', label='Validation Loss',  linewidth=2)
 
    best_epoch = val_history.index(min(val_history)) + 1
    plt.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5,
                label=f'Best Model (Epoch {best_epoch})')
 
    plt.title('GAT Training vs Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (Node CE + Edge CE weighted)')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.7)
 
    plot_path = os.path.join(out_dir, 'loss_curve.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f'\nSaved loss curve plot to: {plot_path}')
    plt.close()
 
plot_losses(train_loss_history, val_loss_history, OUT_DIR)
 
 
# FINAL EVALUATION
 
CAT_NAMES = {
    0: 'Caption', 1: 'Picture', 2: 'Table',   3: 'Formula',
    4: 'Section-header', 5: 'Page-footer', 6: 'Page-header', 7: 'Other',
}
CAT_LABELS  = [CAT_NAMES[i] for i in range(8)]
EDGE_LABELS = ['No-relation', 'Caption–Figure', 'Caption–Table']
 
 
def final_evaluation(loader):
    model.eval()

    all_node_preds,  all_node_labels = [], []
    all_edge_preds,  all_edge_labels = [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            z, node_logits, _ = model(batch)

            node_preds = node_logits.argmax(dim=-1)
            all_node_preds.extend(node_preds.cpu().numpy())
            all_node_labels.extend(batch.y_nodes.cpu().numpy())

            if batch.edge_index_targets.shape[1] == 0:
                continue

            edge_logits = model.predict_edges(
                z, batch.edge_index_targets, batch.feat_geom
            )
            edge_preds = edge_logits.argmax(dim=-1)
            all_edge_preds.extend(edge_preds.cpu().numpy())
            all_edge_labels.extend(batch.y_edges.cpu().numpy())

    # ── Node report ──────────────────────────────────────────────────────────
    print('\n══════════════════════════════════════════')
    print('  NODE CLASSIFICATION')
    print('══════════════════════════════════════════')
    print(classification_report(
        all_node_labels, all_node_preds,
        target_names=CAT_LABELS, labels=list(range(8)), zero_division=0,
    ))
    print('Confusion matrix:')
    print(confusion_matrix(all_node_labels, all_node_preds, labels=list(range(8))))

    # ── Edge report ───────────────────────────────────────────────────────────
    print('\n══════════════════════════════════════════')
    print('  EDGE PREDICTION (3-class: no-rel / cap-fig / cap-tab)')
    print('══════════════════════════════════════════')

    if len(all_edge_preds) == 0:
        print('  ⚠ Žiadne kandidátske hrany v test sete (žiadne Caption × Partner páry).')
        print('  Skontroluj či test_graphs obsahujú y_edges s hodnotami 1 alebo 2.')
        _debug_edge_stats(loader)
        return

    print(classification_report(
        all_edge_labels, all_edge_preds,
        target_names=EDGE_LABELS, labels=[0, 1, 2], zero_division=0,
    ))
    print('Confusion matrix:')
    print(confusion_matrix(all_edge_labels, all_edge_preds, labels=[0, 1, 2]))

    binary_pred  = [1 if p > 0 else 0 for p in all_edge_preds]
    binary_label = [1 if l > 0 else 0 for l in all_edge_labels]
    print('\n── Pozitívne hrany spolu (cap-fig + cap-tab vs. no-rel) ──')
    print(classification_report(
        binary_label, binary_pred,
        target_names=['No-relation', 'Linked'], zero_division=0,
    ))
 
    # ── Node report ──────────────────────────────────────────────────────────
    print('\n══════════════════════════════════════════')
    print('  NODE CLASSIFICATION')
    print('══════════════════════════════════════════')
    print(classification_report(
        all_node_labels, all_node_preds,
        target_names=CAT_LABELS,
        labels=list(range(8)),
        zero_division=0,
    ))
    print('Confusion matrix:')
    print(confusion_matrix(all_node_labels, all_node_preds, labels=list(range(8))))
 
    # ── Edge report — 3 classes directly ────────────────────────────────────────
    print('\n══════════════════════════════════════════')
    print('  EDGE PREDICTION (3-class: no-rel / cap-fig / cap-tab)')
    print('══════════════════════════════════════════')
    print(classification_report(
        all_edge_labels, all_edge_preds,
        target_names=EDGE_LABELS,
        labels=[0, 1, 2],
        zero_division=0,
    ))
    print('Confusion matrix:')
    print(confusion_matrix(all_edge_labels, all_edge_preds, labels=[0, 1, 2]))
 
    # ── Precision/Recall len for positive edges ──────────────
    print('\n── Positive edges in total (cap-fig + cap-tab vs. no-rel) ──')
    binary_pred  = [1 if p > 0 else 0 for p in all_edge_preds]
    binary_label = [1 if l > 0 else 0 for l in all_edge_labels]
    print(classification_report(
        binary_label, binary_pred,
        target_names=['No-relation', 'Linked'],
        zero_division=0,
    ))
 
 
# Load best values before evaluation
model.load_state_dict(
    torch.load(os.path.join(OUT_DIR, 'best_model.pt'), weights_only=True)
)
final_evaluation(test_loader)

USING: cpu

Saving checkpoints to: models/gat_graph_text/run_yolo_augmentation6

Epoch 001 | train: 0.8114 | val: 0.3009 | ✓ NEW BEST
Epoch 002 | train: 0.4113 | val: 0.2667 | ✓ NEW BEST
Epoch 003 | train: 0.3695 | val: 0.2301 | ✓ NEW BEST
Epoch 004 | train: 0.3164 | val: 0.1990 | ✓ NEW BEST
Epoch 005 | train: 0.2821 | val: 0.1839 | ✓ NEW BEST
Epoch 006 | train: 0.2642 | val: 0.1691 | ✓ NEW BEST
Epoch 007 | train: 0.2515 | val: 0.1776 | worse (1/7)
Epoch 008 | train: 0.2415 | val: 0.1702 | worse (2/7)
Epoch 009 | train: 0.2277 | val: 0.1663 | ✓ NEW BEST
Epoch 010 | train: 0.2215 | val: 0.1560 | ✓ NEW BEST
Epoch 011 | train: 0.2151 | val: 0.1489 | ✓ NEW BEST
Epoch 012 | train: 0.2094 | val: 0.1504 | worse (1/7)
Epoch 013 | train: 0.2107 | val: 0.1549 | worse (2/7)
Epoch 014 | train: 0.2020 | val: 0.1542 | worse (3/7)
Epoch 015 | train: 0.2045 | val: 0.1382 | ✓ NEW BEST
Epoch 016 | train: 0.1995 | val: 0.1407 | worse (1/7)
Epoch 017 | train: 0.1960 | val: 0.1485 | worse (2/7)
Epoch 018 |

# Comparison - yolo baseline and heuristics

In [79]:
def final_evaluation(loader):
    model.eval()

    all_node_preds,  all_node_labels  = [], []
    all_yolo_preds                    = []
    all_edge_preds,  all_edge_labels  = [], []
    all_heu_preds,   all_heu_labels   = [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            z, node_logits, _ = model(batch)

            node_preds = node_logits.argmax(dim=-1)
            all_node_preds.extend(node_preds.cpu().numpy())
            all_node_labels.extend(batch.y_nodes.cpu().numpy())

            YOLO_TO_GAT = {
                0: 0,   # YOLO Caption (0) -> GAT Caption (0)
                6: 1,   # YOLO Picture (6) -> GAT Picture (1)
                8: 2,   # YOLO Table (8) -> GAT Table (2)
                2: 3,   # YOLO Formula (2) -> GAT Formula (3)
                7: 4,   # YOLO Section-header (7) -> GAT Section-header (4)
                4: 5,   # YOLO Page-footer (4) -> GAT Page-footer (5)
                5: 6,   # YOLO Page-header (5) -> GAT Page-header (6)
            }
 
            yolo_argmax = batch.feat_yolo.argmax(dim=-1).cpu().numpy()
            yolo_gat = [YOLO_TO_GAT.get(int(y), 7) for y in yolo_argmax]
            all_yolo_preds.extend(yolo_gat)

            if batch.edge_index_targets.shape[1] == 0:
                continue

            # ── GAT hrany ────────────────────────────────────────────────────
            edge_logits = model.predict_edges(
                z, batch.edge_index_targets, batch.feat_geom
            )
            edge_preds = edge_logits.argmax(dim=-1)
            all_edge_preds.extend(edge_preds.cpu().numpy())
            all_edge_labels.extend(batch.y_edges.cpu().numpy())

            ptr = batch.ptr  # [num_graphs+1] — kde začína každý graf
            for g in range(batch.num_graphs):
                start, end = ptr[g].item(), ptr[g + 1].item()

                local_cls    = node_preds[start:end]
                local_geom   = batch.feat_geom[start:end]
    
                          (batch.edge_index_targets[0] <  end)
                local_ei  = batch.edge_index_targets[:, ei_mask] - start
                local_lab = batch.y_edges[ei_mask]

                if local_ei.shape[1] == 0:
                    continue

                # Heuristika: for each caption find closest partner based on centroid
                cap_idx = (local_cls == 0).nonzero(as_tuple=True)[0]
                fig_idx = (local_cls == 1).nonzero(as_tuple=True)[0]
                tab_idx = (local_cls == 2).nonzero(as_tuple=True)[0]

                heu_pos: dict[tuple, int] = {}

                for partner_idx, edge_type in [
                    (fig_idx, 1), (tab_idx, 2)
                ]:
                    partners = partner_idx
                    if len(cap_idx) == 0 or len(partners) == 0:
                        continue
                    
                    # distance bet cent
                    cap_xy = local_geom[cap_idx, 4:6]   # [C, 2]
                    par_xy = local_geom[partners, 4:6]   # [P, 2]
                    # [C, P] euclidean distance
                    diff   = cap_xy.unsqueeze(1) - par_xy.unsqueeze(0)   # [C, P, 2]
                    dists  = diff.norm(dim=-1)   # [C, P]

                    assigned_caps = set()
                    assigned_pars = set()
                    # Greedy
                    pairs_list = []
                    for ci, c in enumerate(cap_idx.tolist()):
                        for pi, p in enumerate(partners.tolist()):
                            pairs_list.append((dists[ci, pi].item(), c, p, edge_type))
                    pairs_list.sort()

                    for dist, c, p, etype in pairs_list:
                        if c in assigned_caps or p in assigned_pars:
                            continue
                        assigned_caps.add(c)
                        assigned_pars.add(p)
                        heu_pos[(c, p)] = etype

                for i in range(local_ei.shape[1]):
                    s = local_ei[0, i].item()
                    d = local_ei[1, i].item()
                    gt_lbl = local_lab[i].item()
                    heu_lbl = heu_pos.get((s, d), 0)
                    all_heu_preds.append(heu_lbl)
                    all_heu_labels.append(gt_lbl)

    # ══════════════════════════════════════════════════════════════════════════
    # VÝPIS
    # ══════════════════════════════════════════════════════════════════════════

    print('\n══════════════════════════════════════════')
    print('  NODE CLASSIFICATION — YOLO BASELINE')
    print('══════════════════════════════════════════')
    print(classification_report(
        all_node_labels, all_yolo_preds,
        target_names=CAT_LABELS, labels=list(range(8)), zero_division=0,
    ))

    print('\n══════════════════════════════════════════')
    print('  NODE CLASSIFICATION — GAT (reclassification)')
    print('══════════════════════════════════════════')
    print(classification_report(
        all_node_labels, all_node_preds,
        target_names=CAT_LABELS, labels=list(range(8)), zero_division=0,
    ))

    # Delta per trieda
    print('  Δ F1 (GAT − YOLO) for class:')
    from sklearn.metrics import f1_score
    yolo_f1s = f1_score(all_node_labels, all_yolo_preds, labels=list(range(8)),
                        average=None, zero_division=0)
    gat_f1s  = f1_score(all_node_labels, all_node_preds, labels=list(range(8)),
                        average=None, zero_division=0)
    for i, name in enumerate(CAT_LABELS):
        delta = gat_f1s[i] - yolo_f1s[i]
        arrow = '▲' if delta > 0.001 else ('▼' if delta < -0.001 else '─')
        print(f'  {name:<16} YOLO={yolo_f1s[i]:.3f}  GAT={gat_f1s[i]:.3f}  {arrow} {delta:+.3f}')

    print('\n══════════════════════════════════════════')
    print('  EDGE — GAT (3-class)')
    print('══════════════════════════════════════════')
    if all_edge_preds:
        print(classification_report(
            all_edge_labels, all_edge_preds,
            target_names=EDGE_LABELS, labels=[0, 1, 2], zero_division=0,
        ))
    else:
        print('  ⚠ No edge predictions.')

    print('\n══════════════════════════════════════════')
    print('  EDGE — HEURISTIC (greedy nearest centroid)')
    print('══════════════════════════════════════════')
    if all_heu_preds:
        print(classification_report(
            all_heu_labels, all_heu_preds,
            target_names=EDGE_LABELS, labels=[0, 1, 2], zero_division=0,
        ))

        # Binary comparison GAT vs heuristic
        print('  ── Binárne (linked vs no-rel) ────────────────')
        from sklearn.metrics import f1_score as sk_f1
        gat_bin_f1 = sk_f1(
            [1 if l > 0 else 0 for l in all_edge_labels],
            [1 if p > 0 else 0 for p in all_edge_preds],
            pos_label=1, zero_division=0,
        )
        heu_bin_f1 = sk_f1(
            [1 if l > 0 else 0 for l in all_heu_labels],
            [1 if p > 0 else 0 for p in all_heu_preds],
            pos_label=1, zero_division=0,
        )
        print(f'  GAT F1 (linked):       {gat_bin_f1:.3f}')
        print(f'  Heuristika F1 (linked):{heu_bin_f1:.3f}')
        print(f'  Δ GAT − Heuristika:   {gat_bin_f1 - heu_bin_f1:+.3f}')
    else:
        print('  ⚠ No heuristic predictions.')
model.load_state_dict(
    torch.load(os.path.join(OUT_DIR, 'best_model.pt'), weights_only=True)
)
final_evaluation(test_loader)


══════════════════════════════════════════
  NODE CLASSIFICATION — YOLO BASELINE
══════════════════════════════════════════
                precision    recall  f1-score   support

       Caption       0.96      0.96      0.96      3809
       Picture       0.99      0.97      0.98      5238
         Table       0.98      0.96      0.97      2625
       Formula       0.98      0.99      0.99      1968
Section-header       0.93      0.97      0.95     10204
   Page-footer       0.98      0.98      0.98      5522
   Page-header       0.95      0.89      0.92      3859
         Other       0.99      0.99      0.99     50813

      accuracy                           0.98     84038
     macro avg       0.97      0.96      0.97     84038
  weighted avg       0.98      0.98      0.98     84038


══════════════════════════════════════════
  NODE CLASSIFICATION — GAT (reklasifikácia)
══════════════════════════════════════════
                precision    recall  f1-score   support

       Capt